# BERT on IMDB: Pretrain, Then Fine-Tune
**COMP 395 — Deep Learning**

Same 12,000 training reviews and 2,000 test reviews as your RNN assignment. Different model: a tiny **pretrained** BERT.

| Part | What you see |
|---|---|
| 1 | How a review becomes token ids |
| 2 | Fine-tune it on IMDB and compare with your RNN and LSTM |
| 3 | Try your own reviews |

**All the code is written for you.** Run the cells in order. The only thing you type is your own reviews at the end.

Needs one package: `pip install transformers`. The first run downloads the model (about 18 MB).

In [1]:
import os, time, tarfile, urllib.request
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

MODEL = "google/bert_uncased_L-2_H-128_A-2"     # BERT-Tiny: 2 layers, 128 numbers per word

device: cuda


In [2]:
def download_imdb(cache_dir='./data'):
    '''Same loader as the RNN assignment: reuses ./data/imdb.pt if it is already there.'''
    os.makedirs(cache_dir, exist_ok=True)
    cached = os.path.join(cache_dir, 'imdb.pt')
    if os.path.exists(cached):
        return torch.load(cached)
    tar_path = os.path.join(cache_dir, 'aclImdb_v1.tar.gz')
    if not os.path.exists(tar_path):
        print("Downloading IMDB (~80 MB)...")
        urllib.request.urlretrieve('https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz', tar_path)
    data = {'train': [], 'test': []}
    with tarfile.open(tar_path, 'r:gz') as tar:
        for m in tar.getmembers():
            parts = m.name.split('/')
            if len(parts) == 4 and parts[1] in data and parts[2] in ('pos', 'neg'):
                text = tar.extractfile(m).read().decode('utf-8')
                data[parts[1]].append((1 if parts[2] == 'pos' else 0, text))
    torch.save((data['train'], data['test']), cached)
    return data['train'], data['test']

all_train, all_test = download_imdb()
rng = np.random.default_rng(0)                  # same seed, same reviews as the RNN assignment
train_raw = [all_train[i] for i in rng.permutation(len(all_train))[:12000]]
test_raw  = [all_test[i]  for i in rng.permutation(len(all_test))[:2000]]
print(len(train_raw), "train reviews,", len(test_raw), "test reviews")

12000 train reviews, 2000 test reviews


---
## Part 1 — From review to token ids

Same first step as the embedding slide: split, look up an id, look up that id's row of the embedding table. BERT splits rare words into pieces (`##` means "glued to the piece before") and adds two special tokens:
- `[CLS]` at the front. Its output vector is what we'll classify.
- `[SEP]` at the end.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

review = "This movie was not very good, honestly unwatchable."
ids = tokenizer(review)["input_ids"]
for i in ids:
    print(f"{i:6d}  {tokenizer.convert_ids_to_tokens(i)}")

   101  [CLS]
  2023  this
  3185  movie
  2001  was
  2025  not
  2200  very
  2204  good
  1010  ,
  9826  honestly
  4895  un
 18866  ##watch
  3085  ##able
  1012  .
   102  [SEP]


---
## Part 2 — Fine-tune on IMDB

We keep BERT's pretrained body and add one new layer on top of `[CLS]`: 128 numbers in, 2 scores out (negative, positive). Then we train **everything** for a few passes with a small learning rate.

When it loads, Transformers prints a report. `classifier.weight` and `classifier.bias` are listed as missing: that's the new layer, created from scratch. `cls.predictions...` may be listed as unexpected: that's the fill-in-the-blank layer from pretraining, which we throw away. Both are expected.

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
n = sum(p.numel() for p in model.parameters())
print(f"{n:,} parameters; the new layer is {model.classifier}")

W1102 13:42:01.894000 3704 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4,386,178 parameters; the new layer is Linear(in_features=128, out_features=2, bias=True)


Tokenize every review. BERT reads at most 512 tokens; we keep the **last** 256, like your RNN kept the last 200 words (reviews usually end with the verdict).

In [5]:
MAX_LEN = 256
tokenizer.truncation_side = "left"              # cut from the front, keep the end

def encode(raw):
    enc = tokenizer([text for _, text in raw], truncation=True, max_length=MAX_LEN,
                    padding="max_length", return_tensors="pt")
    labels = torch.tensor([label for label, _ in raw])
    return torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

train_loader = torch.utils.data.DataLoader(encode(train_raw), batch_size=32, shuffle=True)
test_loader  = torch.utils.data.DataLoader(encode(test_raw),  batch_size=64)

@torch.no_grad()
def accuracy(loader):
    model.eval(); correct = 0
    for ids, mask, y in loader:
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        correct += (logits.argmax(-1).cpu() == y).sum().item()
    return correct / len(loader.dataset)

print(f"test accuracy before fine-tuning: {accuracy(test_loader):.1%}   (the new layer is random)")

test accuracy before fine-tuning: 49.5%   (the new layer is random)


Now train: 3 passes over the 12,000 reviews. On a laptop this takes a few minutes; the cell prints the time per pass.

In [6]:
EPOCHS = 3
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
epoch_metrics = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    total = 0
    for ids, mask, y in train_loader:
        loss = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    if device == "cuda":
        torch.cuda.synchronize()
    train_seconds = time.perf_counter() - t0
    test_accuracy = accuracy(test_loader)
    elapsed = time.perf_counter() - t0
    epoch_metrics.append({"epoch": epoch, "loss": total / len(train_loader),
                          "test_accuracy": test_accuracy, "train_seconds": train_seconds,
                          "train_and_eval_seconds": elapsed})
    print(f"epoch {epoch}: loss {total / len(train_loader):.3f}   "
          f"test accuracy {test_accuracy:.2%}   ({elapsed:.1f} s including evaluation)")

print(f"Fine-tuning time (training only): {sum(m['train_seconds'] for m in epoch_metrics):.1f} s")
print(f"Training + epoch evaluations: {sum(m['train_and_eval_seconds'] for m in epoch_metrics):.1f} s")


epoch 1: loss 0.480   test accuracy 83.80%   (9.7 s including evaluation)


epoch 2: loss 0.294   test accuracy 85.95%   (8.5 s including evaluation)


epoch 3: loss 0.196   test accuracy 85.90%   (4.7 s including evaluation)
Fine-tuning time (training only): 22.2 s
Training + epoch evaluations: 23.0 s


**Question 1** — Fill in the table with your numbers from the RNN assignment and from this notebook.

| model | test accuracy | training time |
|---|---|---|
| RNN (10 epochs, original assignment) | 73.90% | Not recorded separately; see timing note below |
| LSTM (10 epochs, original assignment) | 84.15% | Not recorded separately; see timing note below |
| BERT-Tiny, pretrained + fine-tuned (3 epochs) | 85.90% | 22.2 s for training; 23.0 s including epoch evaluations |

**Answer:** BERT-Tiny's final accuracy is 12.00 percentage points higher than the RNN's and 1.75 points higher than the LSTM's. Its epoch accuracies were 83.80%, 85.95%, and 85.90%. The table uses the final epoch rather than selecting the best epoch on the test set.

The RNN/LSTM accuracies come from the saved original last-200-word experiments in `Lecture 6-RNN/Day 2-Sentiment/Sentiment.ipynb`, checked against `data/part4_trained_models.pt`. Those models were not retrained for this comparison. The cached IMDB reviews and NumPy split seed are the same in both assignments.

**Timing note:** The old notebook did not time RNN and LSTM separately. Its shared training cell ran for about **45.7 s total**, including both models, evaluations, and plotting; this cannot be split into valid per-model training times. BERT timing was measured in this run on an NVIDIA GeForce RTX 4060 Laptop GPU, with CUDA synchronization at the training boundaries. It excludes downloads, tokenization, and the original pretraining cost. The different epoch counts and timing boundaries prevent a direct speed comparison.


---
## Part 3 — Your own reviews

Write a few reviews of your own and run the cell. Try to fool it: sarcasm, a twist ending, *not bad*.

In [7]:
my_reviews = [
    "Not very good. I wanted to leave after twenty minutes.",
    "I expected to hate it, but it was wonderful.",
    "Not bad at all. The characters felt real, and I would watch it again.",
    "Great, another two hours of wooden acting and a plot that goes nowhere.",
    "The opening was slow and dull, but the final act made the whole movie worth watching.",
    "It started with a clever idea, but the ending ruined everything I liked about it.",
    "The actors tried their best. Unfortunately, that is the only nice thing I can say.",
    "I cannot say I did not enjoy it, although I would not call it a masterpiece.",
]
model.eval()
with torch.no_grad():
    enc = tokenizer(my_reviews, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits, dim=-1)[:, 1]
for r, p in zip(my_reviews, probs):
    print(f"{p:.2f} positive   {r}")

0.20 positive   Not very good. I wanted to leave after twenty minutes.
0.98 positive   I expected to hate it, but it was wonderful.
0.04 positive   Not bad at all. The characters felt real, and I would watch it again.
0.03 positive   Great, another two hours of wooden acting and a plot that goes nowhere.
0.01 positive   The opening was slow and dull, but the final act made the whole movie worth watching.
0.18 positive   It started with a clever idea, but the ending ruined everything I liked about it.
0.11 positive   The actors tried their best. Unfortunately, that is the only nice thing I can say.
0.91 positive   I cannot say I did not enjoy it, although I would not call it a masterpiece.


**Observations from the custom reviews:** Using 0.5 as the positive/negative cutoff, BERT recognizes the sarcastic "Great, another two hours..." review as negative (0.03 positive), and it recognizes the disappointment in "the ending ruined everything" (0.18). However, it misses the positive meaning of "Not bad at all" (0.04) and the positive final turn in "the final act made the whole movie worth watching" (0.01). The double-negative review receives 0.91 positive, which fits its mildly positive meaning, although the review is mixed.

These are illustrative, AI-assisted examples, not an additional labeled test set. A high positive probability is a model score, not proof that its interpretation is correct.

**Question 2** — Your LSTM and BERT-Tiny saw the **same** 12,000 labeled reviews. Why does BERT do better?

**Answer:** The labeled training set is the same, but their starting points are different. BERT-Tiny already learned language patterns through pretraining on a much larger text corpus. Fine-tuning adapts those learned representations to sentiment. The LSTM also used pretrained GloVe embeddings, but its recurrent layers and classifier learned the task from the IMDB reviews; it did not start with a pretrained contextual sentence encoder.

BERT's bidirectional self-attention lets a word's representation use context on both sides and connect directly with other positions in the retained review. This can help with negation and relationships between distant words. Its WordPiece tokenizer also represents many unfamiliar words as smaller pieces. Google's [BERT-Tiny model card](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2) describes the pretrained model and its WordPiece masking setup.

In this run, BERT reaches 85.90% versus the LSTM's 84.15%. Pretraining and contextual attention help explain that result, but this experiment does not isolate their individual effects: model size, tokenizer, input length, optimizer settings, and epoch count also differ. One run does not prove BERT will always win, and the custom reviews show that it still makes mistakes.


### Execution and assistance note

All seven code cells were executed in order using Python 3.14.5, PyTorch 2.10.0+cu130, Transformers 5.22.1, and CUDA. The supplied model, data sizes, batch sizes, learning rate, and three-epoch schedule were retained. A fixed PyTorch seed and explicit timing were added for this run. Small numerical differences may occur when rerunning on other hardware or library versions.

AI assistance was used to prepare the additional reviews, draft the explanations, execute the notebook, and verify its saved outputs. The notebook contains saved results; restarting the kernel requires running the prerequisite cells again. To rerun locally, select `Lecture 8-BERT/.venv/Scripts/python.exe` as the interpreter (it reuses the course PyTorch environment and adds Transformers). On another machine, use an environment with PyTorch, NumPy, Transformers, and a Jupyter kernel installed. Model and dataset downloads require internet access when not cached.
